## 0. This checkout, not whatever is installed

`zou_lab_control_v2` is the one entry: importing it puts this checkout's eight layers
on the path, ahead of anything else.  It has to come **first** -- if a `zlc_*` module
was already imported from somewhere else, it refuses out loud rather than leaving two
copies in one kernel.  (If that happens: restart the kernel and run this cell first.)


In [ ]:
import sys
from pathlib import Path

_here = Path.cwd().resolve()
while not (_here / 'zou_lab_control_v2').is_dir() and _here != _here.parent:
    _here = _here.parent
sys.path.insert(0, str(_here))

import zou_lab_control_v2

print('code in use:', zou_lab_control_v2.ROOT)


# zlc_pulse tutorial

This notebook teaches the public sequence/model API first, then uses the real remote server at `127.0.0.1:18861`. The last executable cell always drives SAFE before closing.

In [ ]:
from pathlib import Path
import inspect
import sys
repo = Path.cwd()
if not (repo / 'src').is_dir(): repo = repo.parent
sys.path.insert(0, str(repo / 'src'))
from zlc_pulse import (DEFAULT_HOST, DEFAULT_PORT, MINIMUM_REPEAT_COUNT, MemoryRegisterTransport,
    TIME_UNIT_CHOICES, TIME_UNIT_TO_NS, align_to_grid, resolve_scan_point, scan_columns_for, scan_table_template, validate_scan_table, scan_rows_to_wire, scan_rows_from_wire, PULSE_TREE_FORMAT, sequence_to_tree, sequence_from_tree, AnalogStep, OutputDelay, PulseFieldRef,
    PulsePeriod, PulsePortSpec, PulseSequence, PulseSlot, PulseStreamer, PulseTarget,
    RemoteError, RemotePulseStreamer, RepeatRegion, __version__, compile_sequence, connect, load_streamer_config,
    pulse_target_from_xdc, serve)
print('zlc_pulse', __version__)


## 1. Build a logical target

`PulseTarget` maps raw lanes to logical digital and DAC ports. The compiler uses this mapping; it does not know about a GUI or a run identity.

In [ ]:
target = PulseTarget(
    lanes=('trigger', 'dac0', 'dac1'),
    ports=(PulsePortSpec('trigger', 'digital', ('trigger',)),
           PulsePortSpec('dac', 'dac', ('dac0', 'dac1'), bus_index=0)),
)
print('lanes:', target.raw_lanes)
print('ports:', [(port.key, port.kind) for port in target.ports])

## 2. Describe periods, slots, delays, and repeats

A `PulseSlot` makes one duration an application value after compilation. `AnalogStep` describes a DAC edge or ramp; `OutputDelay` and `RepeatRegion` stay inside the sequence timeline. A `RepeatRegion` never means repeated cycles, scan sweeps, or Dataset repeats; those remain separate execution/data facts.

In [ ]:
duration_slot = PulseSlot('duration', PulseFieldRef('duration', 'high'), 'ns', 'high_ticks')
sequence = PulseSequence(
    name='offline-demo', target=target, time_step_ns=20,
    periods=(PulsePeriod('high', 20, 'ns', (1, 0, 0), (AnalogStep('dac', 'ramp', 1),)),
             PulsePeriod('low', 20, 'ns', (0, 0, 0), (AnalogStep('dac', 'edge', -2),))),
    slots=(duration_slot,), delays=(OutputDelay('trigger', 0, 'ns'),),
    # MINIMUM_REPEAT_COUNT is the smallest count that IS a repeat: playing
    # the periods once is the sequence itself, so the model refuses it.
    repeat=RepeatRegion('high', 'low', MINIMUM_REPEAT_COUNT),
)
print('sequence:', sequence.name, 'periods=', len(sequence.periods), 'slot=', duration_slot.slot_id)
# A pulse has a persisted form, which is what an editor's Save writes and
# its Load reads.  The tree goes back through the model's own constructors,
# so a file describing an illegal pulse is refused rather than becoming one.
tree = sequence_to_tree(sequence)
reopened = sequence_from_tree(tree)
print('format:', tree['format'], PULSE_TREE_FORMAT, '| same pulse:', reopened == sequence)
# A pulse has a persisted form: an editor writes this and reads it back,
# and the tree goes through the model's own constructors on the way in,
# so a file describing an illegal pulse is refused rather than becoming one.


## 3. Use the shipped geometry and clock

`load_streamer_config()` returns the geometry value used by the compiler and the server. Users consume the returned `params` object rather than constructing a geometry type themselves.

In [ ]:
config = load_streamer_config()
geometry = config['params']
clock_hz = config['clock_hz']
print('clock:', int(clock_hz), 'channels:', geometry.channel_count, 'DAC buses:', geometry.bus_count)

## 4. Load the explicit board target

`streamer_config.json` `board.lanes` is the mapping authority. `pulse_target_from_xdc()` generates the target from those explicit indices and rejects any mismatch in the checked-in XDC port/pin or RTL top projection; XDC declaration order has no meaning.

In [ ]:
board_target = pulse_target_from_xdc()
print('Board lanes:', len(board_target.raw_lanes), 'ports:', len(board_target.ports),
      'pinned lanes:', len(board_target.package_pins))

## 5. Compile once

`compile_sequence()` returns a `CompiledProgram`; inspect its edge rows and DAC segments, then keep the returned object for `load()`.

In [ ]:
program = compile_sequence(sequence, geometry, clock_hz)
print('CompiledProgram edges:', len(program.ticks), 'DAC segments:', len(program.bus_segments),
      'slot kinds:', program.slot_kinds)
print('edge ticks:', program.ticks, 'masks:', program.masks)

## 6. See the local device and transport choices

`PulseStreamer` is the local register device. The offline memory transport exercises the same register contract without opening hardware.

In [ ]:
memory = MemoryRegisterTransport(geom=geometry)
local_device = PulseStreamer(memory, geometry, clock_hz, target=board_target)
print('local device:', type(local_device).__name__, 'opened=', local_device.snapshot()['opened'])
print('transport cadence:', memory.observer_interval)

## 7. Recognize the three operator-facing error layers

`RemoteError` reports an error returned by the pulse server.

In [ ]:
error_examples = (RemoteError('server', 'busy'),)
for error in error_examples:
    print('error layer:', type(error).__name__, 'message:', str(error))

## 8. Understand the remote connection surface

`RemotePulseStreamer` adds only connection lifecycle helpers. `connect()` is the convenience constructor; `serve()` is the server entry point used by `run_server.bat`.

In [ ]:
remote_preview = RemotePulseStreamer(DEFAULT_HOST, DEFAULT_PORT)
print('remote endpoint:', remote_preview.host + ':' + str(remote_preview.port))
print('server signature has streamer/host/port:',
      all(name in inspect.signature(serve).parameters for name in ('streamer', 'host', 'port')))


## 8b. Units, value tables, and a board with no wire

Four questions a host asks that the model owns the answers to: which units a duration may be
written in and what each is worth in nanoseconds; which columns a bound pulse's scan table has;
what a starting table looks like; and whether a table is legal.  `MemoryRegisterTransport` is
the same command sequence with the wire substituted -- how a board is driven with no board.


In [ ]:
print('time units:', TIME_UNIT_CHOICES)
print('one us in ns:', TIME_UNIT_TO_NS['us'])
scan_columns = scan_columns_for(sequence)
# A column speaks the unit the EDITOR shows for the field it scans -- a
# signed DAC code, a duration in its period's own unit -- and carries how
# to reach the wire, so nobody writing a table has to know two number
# systems for one output.
print('scan columns:', [(c.name, c.unit, c.lo, c.hi, c.wire_scale, c.wire_offset)
                        for c in scan_columns])
print('starter program:', scan_table_template('column_stack', scan_columns).splitlines()[0])
author_rows = validate_scan_table(((21,), (22,)), scan_columns)
print('a legal table:', author_rows)
# The one crossing, and it goes both ways: what a slot holds, and what its
# author wrote.
on_the_wire = scan_rows_to_wire(author_rows, scan_columns)
print('what the slots hold:', on_the_wire)
print('and back again:', scan_rows_from_wire(on_the_wire, scan_columns))
try:
    validate_scan_table(((21, 99),), scan_columns)
except ValueError as error:
    print('and an illegal one:', error)
offline_transport = MemoryRegisterTransport()
print('a board with no wire:', type(offline_transport).__name__)


### Holding one point of a scan

A held point is an ordinary pulse carrying that row's numbers -- not a scan
of length one.  `resolve_scan_point` writes the row into the fields its slots
name and hands back a sequence with no slots left, which loads and runs like
any other pulse.  Saying it the other way round steers the board into a
one-point scan, which is a corner nothing else asks it for.


In [ ]:
row = [column.lo for column in scan_columns]
held = resolve_scan_point(sequence, row)
print('row held:', row)
print('slots left after resolving:', held.slots)
print('periods now:', [(p.period_id, p.duration, p.unit) for p in held.periods])


### Rounding a duration onto the clock

`exact_ticks` asks whether a value is legal.  `align_to_grid` asks which
legal value was meant -- the question an editor has while someone is still
typing.  Round with it before authoring, and the strict check still stands
guard over what gets written down.


In [ ]:
# 1 us is exactly 50 ticks of a 20 ns clock; 37 ns falls between ticks.
print('37 ns snaps to', align_to_grid(37, 'ns', 20, 'duration'), 'ns')
print('1 us is already on the grid:', align_to_grid(1, 'us', 20, 'duration'), 'us')


## 9. Prepare the real board program

This cell derives every lane and DAC port from the explicit board manifest and validates its XDC/RTL projections. The next cells use the local server endpoint printed by `run_server.bat`; there is no environment-variable backend switch in the notebook.

In [ ]:
hardware_config = load_streamer_config()
hardware_geometry = hardware_config['params']
hardware_clock_hz = hardware_config['clock_hz']
hardware_target = pulse_target_from_xdc()
print('hardware target:', len(hardware_target.raw_lanes), 'lanes,',
      sum(port.kind == 'dac' for port in hardware_target.ports), 'DAC buses')

## 10. Make the requested all-channel 1 µs loop

Every digital lane is high for 1 µs and low for 1 µs. Every DAC bus ramps from signed `-max` to `+max` during the high period and returns to `-max` in the low period, so the cycle repeats continuously.

In [ ]:
hardware_lanes = hardware_target.raw_lanes
digital_lanes = {lane for port in hardware_target.ports if port.kind == 'digital' for lane in port.lanes}
hardware_high = tuple(int(lane in digital_lanes) for lane in hardware_lanes)
hardware_low = (0,) * len(hardware_lanes)
hardware_dac_min = -(1 << (hardware_geometry.bus_width - 1))
hardware_dac_max = (1 << (hardware_geometry.bus_width - 1)) - 1
hardware_up = tuple(AnalogStep(port.key, 'ramp', hardware_dac_max) for port in hardware_target.ports if port.kind == 'dac')
hardware_down = tuple(AnalogStep(port.key, 'ramp', hardware_dac_min) for port in hardware_target.ports if port.kind == 'dac')
hardware_slot = PulseSlot('duration', PulseFieldRef('duration', 'high'), 'us', 'high_us')
hardware_sequence = PulseSequence(name='all-channel-1us-loop', target=hardware_target, time_step_ns=20,
    periods=(PulsePeriod('high', 1, 'us', hardware_high, hardware_up),
             PulsePeriod('low', 1, 'us', hardware_low, hardware_down)), slots=(hardware_slot,))
hardware_program = compile_sequence(hardware_sequence, hardware_geometry, hardware_clock_hz)
hardware_host, hardware_port = '127.0.0.1', 18861
hardware_endpoint = f'{hardware_host}:{hardware_port}'
remote = connect(hardware_host, hardware_port, request_timeout=30.0, poll_interval=0.01)
print('hardware program: 1 us high/low, DAC range', (hardware_dac_min, hardware_dac_max))

## 11. Open the real remote session

Run `bin\run_server.bat` first. A successful `open()` means the server has established its hardware session and verified SAFE readiness; a failure below names the exact endpoint.

In [ ]:
try:
    remote.open()
except (RemoteError, ConnectionError, TimeoutError, OSError) as error:
    raise RuntimeError(f'Start bin\\run_server.bat and wait for HARDWARE CONNECTED/RPC LISTENING at {hardware_endpoint}, then rerun this cell.') from error
print('server CONNECTED:', hardware_endpoint)

## 12. Load the compiled program

`load()` atomically uploads the edge/DAC/delay image together with the complete value table. This slotted program therefore supplies its 50-tick row in the same call.

In [ ]:
remote.load(hardware_program, source=hardware_sequence, rows=((50,),))
print('loaded and initialized:', len(hardware_program.ticks), 'edges, 50-tick (1 us) high duration')

## 13. Read the passive `applied()` echo

The returned object is the device's last applied record, not a trigger schedule or an external run acknowledgement.

In [ ]:
applied_state = remote.applied()
print('applied type:', type(applied_state).__name__,
      'source:', applied_state.source.name if applied_state and applied_state.source else None,
      'rows:', applied_state.rows if applied_state else None,
      'cycles:', applied_state.cycles if applied_state else None)

## 14. Fire once and inspect `DoneReport`

The finite run returns two status samples, two cursor samples, underflow, and the post-terminal tail time.

In [ ]:
remote.fire(cycles=1)
done_report = remote.wait_done(5.0)
print('DoneReport:', {
    'status_first': done_report.status_first if done_report else None, 'status_second': done_report.status_second if done_report else None,
    'cursor_first': done_report.cursor_first if done_report else None, 'cursor_second': done_report.cursor_second if done_report else None,
    'status': done_report.status if done_report else None, 'underflow': done_report.underflow if done_report else None,
    'tail_elapsed': done_report.tail_elapsed if done_report else None})

## 15. Load a different one-row application

A new slot value is a new complete application of the same compiled program. `load(..., rows=...)` replaces the image and values atomically; there is no later mutation call.

In [ ]:
remote.load(hardware_program, source=hardware_sequence, rows=((60,),))
row_state = remote.applied()
remote.fire(cycles=1)
row_done = remote.wait_done(5.0)
print('applied rows:', row_state.rows if row_state else None,
      'fire status:', row_done.status if row_done else None)

## 16. Load a multi-row application and inspect `cursor()`

A value table may contain multiple rows. It is loaded atomically with the program; the finite cycle count says how many source cycles execute, while `cursor()` reports non-blocking board progress.

In [ ]:
scan_rows = ((50,), (52,), (55,), (58,), (60,))
remote.load(hardware_program, source=hardware_sequence, rows=scan_rows)
remote.fire(cycles=len(scan_rows))
cursor_samples = (remote.cursor(), remote.cursor(), remote.cursor())
scan_done = remote.wait_done(5.0)
print('scan rows:', len(scan_rows), 'cursor samples:', cursor_samples,
      'done status:', scan_done.status if scan_done else None)

## 17. Run continuously with guaranteed SAFE cleanup

`cycles=None` is the explicit forever form. This cell waits for Enter while the board runs, but its `finally` block always requests verified SAFE before control returns.

In [ ]:
try:
    remote.load(hardware_program, source=hardware_sequence, rows=scan_rows)
    remote.fire(cycles=None)
    forever_snapshot = remote.snapshot()
    print('continuous:', {key: forever_snapshot[key] for key in ('firing', 'cycles', 'status')})
    input('Press Enter to drive SAFE...')
finally:
    safe_readback = remote.safe()
    print('SAFE stable:', safe_readback.stable)

## 18. Close the SAFE tutorial connection

The preceding `finally` already obtained verified SAFE. `close()` now releases this remote connection before the final cell creates a fresh one.

In [ ]:
remote.close()
print('remote closed after verified SAFE')

## 19. Direct real-hardware scope run — run this cell last

This operator cell connects to the local server, atomically loads a static (unslotted) three-edge program, and uses `cycles=None` while the scope is observed. F15 and M13 carry a 500 kHz square wave (1 µs high, 1 µs low); all four DAC buses run the same signed-range ramp cycle. Press Enter to finish; the `finally` block drives verified SAFE and closes the connection even if the cell is interrupted.

In [ ]:
scope_sequence = PulseSequence(name='all-channel-static-scope-loop', target=hardware_target, time_step_ns=20,
    periods=(PulsePeriod('high', 1, 'us', hardware_high, hardware_up),
             PulsePeriod('low', 1, 'us', hardware_low, hardware_down)))
scope_program = compile_sequence(scope_sequence, hardware_geometry, hardware_clock_hz)
scope_pin_lanes = {pin: lane for lane, pin in hardware_target.package_pins.items() if pin in {'F15', 'M13'}}
scope_pin_bits = tuple(hardware_lanes.index(scope_pin_lanes[pin]) for pin in ('F15', 'M13'))
if scope_program.ticks != (0, 50, 100) or scope_program.slot_count != 0:
    raise RuntimeError(f'scope program is not static 1 us/1 us: ticks={scope_program.ticks}, slots={scope_program.slot_count}')
if any(not (scope_program.masks[0] & (1 << bit)) for bit in scope_pin_bits):
    raise RuntimeError(f'F15/M13 are absent from high mask 0x{scope_program.masks[0]:X}')
remote = connect('127.0.0.1', 18861, request_timeout=30.0, poll_interval=0.01)
remote.open()
try:
    remote.load(scope_program, source=scope_sequence)
    remote.fire(cycles=None)
    scope_snapshot = remote.snapshot()
    print('REAL FPGA FIRING:', scope_snapshot)
    print('scope TTL pins: F15 and M13 = 1 us HIGH / 1 us LOW; DAC buses:',
          tuple(port.key for port in hardware_target.ports if port.kind == 'dac'))
    input('Press Enter to drive SAFE and close...')
finally:
    try:
        scope_safe = remote.safe()
    finally:
        remote.close()
    print('scope run stopped; SAFE stable:', scope_safe.stable)